In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve, auc, roc_curve
from sklearn.utils import shuffle
from datetime import timedelta
from src.common.feature_dtypes import expected_dtypes
from tqdm import tqdm
import random
import os
import boto3
import tempfile
import io

In [36]:

# Define S3 info
bucket = 'kehmisjan2025'

# Initialize boto3 client
s3 = boto3.client('s3')
buffer = io.BytesIO()
s3.download_fileobj(bucket, 'rtc0829.parquet', buffer)
buffer.seek(0)  # Move to the start of the buffer
df = pd.read_parquet(buffer)


In [37]:
# create variables is_linked if ReasonforMissedAppt is not null
df['is_linked'] = df['ReasonforMissedAppt'].notnull()

# get count by is_linked
df['is_linked'].value_counts()

is_linked
False    663624
True      22126
Name: count, dtype: int64

In [38]:
# filter to emr in kenyamer and ecare
df = df[df["emr"].isin(["kenyaemr"])]
df = df.drop(columns=["emr"])

df.columns = df.columns.str.lower().str.replace(" ", "_")

# ensure columns are right dtypes
for col, dtype in expected_dtypes.items():
    if col in df.columns:
        if dtype in [float, "float", "float64", int, "int", "int64"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        else:
            df[col] = df[col].astype(dtype)

In [39]:
cols_to_keep = ['cascadestatus', 'visittype', 'visitby', 'tcareason', 'pregnant',
       'breastfeeding', 
       'stabilityassessment', 'differentiatedcare', 'whostage',
       'adherence', 'sex', 'age', 'maritalstatus', 'educationlevel',
       'occupation', 'bmi', 'regimen_switch',  'is_friday', 'daystonextappointment', 'timeonart',
       'firstvisit', 'lastvd', 'late', 'late14', 'late30',
       'lateness_last3', 'lateness_last5', 'lateness_last10', 'late_last3',
       'late_last5', 'late_last10', 'late14_last3', 'late14_last5',
       'late14_last10', 'late30_last3', 'late30_last5', 'late30_last10',
       'optimizedhivregimen', 'most_recent_vl', 'ahd', 'kephlevel',
       'facilitytypecategory', 'ownertype', 'men_knowledge',
       'women_knowledge', 'men_heardaids', 'men_highrisksex',
       'men_highrisksex_multi', 'men_sexnotwithpartner', 'men_sexpartners',
       'men_nevertested', 'men_testedrecent', 'men_sti', 'women_heardaids',
       'women_highrisksex', 'women_highrisksex_multi',
       'women_sexnotwithpartner', 'women_sexpartners', 'women_nevertested',
       'women_testedrecent', 'women_sti', 'rolling_weighted_noshow',
       'rolling_weighted_dayslate', 'is_linked'

]

In [40]:
# create df_ipw as df subset by cols to keep
print(df.shape)
df_ipw = df[cols_to_keep]
print(df_ipw.shape)

(614102, 87)
(614102, 64)


In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
import numpy as np

exclude_cols = ['is_linked']

# Numeric columns
num_vars = df_ipw.select_dtypes(include=['int32', 'int64', 'float64', 'bool']).columns.tolist()
num_vars = [c for c in num_vars if c not in exclude_cols]

# Convert bool to int
for c in num_vars:
    if df_ipw[c].dtype == 'bool':
        df_ipw[c] = df_ipw[c].astype(int)

# Impute numeric variables safely
num_imputer = SimpleImputer(strategy='median')
num_imputed_array = num_imputer.fit_transform(df_ipw[num_vars])

# Make sure number of columns matches
num_imputed_df = pd.DataFrame(num_imputed_array, columns=num_vars, index=df_ipw.index)
df_ipw[num_vars] = num_imputed_df

# Categorical columns
cat_vars = df_ipw.select_dtypes(include=['object', 'category']).columns.tolist()
cat_vars = [c for c in cat_vars if c not in exclude_cols]

if cat_vars:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    cat_imputed_array = cat_imputer.fit_transform(df_ipw[cat_vars])
    cat_imputed_df = pd.DataFrame(cat_imputed_array, columns=cat_vars, index=df_ipw.index)
    df_ipw[cat_vars] = cat_imputed_df

    # One-hot encode
    df_ipw = pd.get_dummies(df_ipw, columns=cat_vars, drop_first=True)

print(df_ipw.shape)


/tmp/ipykernel_14743/41971533.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ipw[num_vars] = num_imputed_df
/tmp/ipykernel_14743/41971533.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ipw[cat_vars] = cat_imputed_df


(614102, 101)


In [43]:
# x_prop drops is_linked
X_prop = df_ipw.drop(columns=['is_linked'])
y_prop = df_ipw['is_linked'].astype(int)
print(X_prop.shape, y_prop.shape)

(614102, 100) (614102,)


In [ ]:
# Step 1: Fit propensity model
from sklearn.linear_model import LogisticRegression
prop_model = LogisticRegression(max_iter=5000)
prop_model.fit(X_prop, y_prop)

# Step 2: Predict propensity scores for *all* patients
df_ipw['propensity'] = prop_model.predict_proba(X_prop)[:,1]

# Step 3: Calculate stabilized weights
p_linked = df_ipw['is_linked'].mean()   # overall fraction linked
df_ipw['stabilized_weight'] = np.where(
    df_ipw['is_linked'] == 1,
    p_linked / df_ipw['propensity'],
    (1 - p_linked) / (1 - df_ipw['propensity'])
)

# Step 4: Cap extreme weights
df_ipw['stabilized_weight'] = df_ipw['stabilized_weight'].clip(upper=10)

# hstack two columns from df - ReasonforMissedAppt and TracingType
df_ipw['reasonformissedappt, tracingtype'] = df['reasonformissedappt'].astype(str) + "_" + df['tracingtype'].astype(str)

# Step 5: Keep only linked patients for downstream modeling
linked = df_ipw[df_ipw['is_linked'] == 1].copy()

In [ ]:
columns_to_drop = ['key', 'visitdate', 'nad', 'nad_imputation_flag',
                   'sitecode', 'iit', 'pregnant_missing', 'breastfeeding_missing',
                   'startartdate', 'month', 'dayofweek', 'timeatfacility',
                   'code', 'county', ]

Index(['key', 'visitdate', 'nad', 'nad_imputation_flag', 'sitecode', 'iit',
       'cascadestatus', 'visittype', 'visitby', 'tcareason', 'pregnant',
       'pregnant_missing', 'breastfeeding', 'breastfeeding_missing',
       'stabilityassessment', 'differentiatedcare', 'whostage', 'adherence',
       'sex', 'age', 'maritalstatus', 'educationlevel', 'occupation', 'bmi',
       'regimen_switch', 'startartdate', 'month', 'dayofweek', 'is_friday',
       'daystonextappointment', 'timeonart', 'timeatfacility', 'firstvisit',
       'lastvd', 'late', 'late14', 'late30', 'lateness_last3',
       'lateness_last5', 'lateness_last10', 'late_last3', 'late_last5',
       'late_last10', 'late14_last3', 'late14_last5', 'late14_last10',
       'late30_last3', 'late30_last5', 'late30_last10', 'optimizedhivregimen',
       'most_recent_vl', 'ahd', 'code', 'kephlevel', 'county',
       'facilitytypecategory', 'ownertype', 'men_knowledge', 'women_knowledge',
       'men_heardaids', 'men_highrisksex', 'men

In [47]:
linked.shape

(22126, 89)